In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# plt.rcParams()
# plt.rcParams['text.usetex'] = True 
def compute_calibration_bins(confidence, correctness, nbins=10):
    """
    Compute bin-wise confidence and accuracy means, and ECE for a calibration plot.

    Args:
        confidence: Array of confidence scores (0 to 1)
        correctness: Array of correctness values (0 or 1)
        nbins: Number of bins for binning confidence scores

    Returns:
        tuple: (bin_conf_means, bin_acc_means, ece)
            - bin_conf_means: Mean confidence per bin
            - bin_acc_means: Mean accuracy per bin
            - ece: Expected Calibration Error
    """
    bins = np.linspace(0, 1, nbins + 1)
    bin_indices = np.digitize(confidence, bins) - 1

    bin_conf_sums = np.zeros(nbins, dtype=float)
    bin_acc_sums = np.zeros(nbins, dtype=float)
    bin_counts = np.zeros(nbins, dtype=int)

    for i in range(len(confidence)):
        b = bin_indices[i]
        if 0 <= b < nbins:
            bin_conf_sums[b] += confidence[i]
            bin_acc_sums[b] += correctness[i]
            bin_counts[b] += 1

    bin_conf_means = []
    bin_acc_means = []
    ece = 0.0
    total = float(len(confidence))
    for b in range(nbins):
        if bin_counts[b] > 0:
            mean_conf = bin_conf_sums[b] / bin_counts[b]
            mean_acc = bin_acc_sums[b] / bin_counts[b]
            bin_conf_means.append(mean_conf)
            bin_acc_means.append(mean_acc)
            fraction_b = bin_counts[b] / total
            ece += fraction_b * abs(mean_acc - mean_conf)
    return bin_conf_means, bin_acc_means, ece

def plot_multiple_calibration_curve(
    df, confidence_cols, correctness_col, additional_lines=None,
    nbins=10, plot_title='Calibration Plot', save_path=None
):
    """
    Plot multiple reliability diagrams (confidence vs. accuracy) on the same plot.

    Args:
        df: DataFrame with confidence and correctness data
        confidence_cols: List of column names in df for confidence scores
        correctness_col: Column name in df for correctness (0 or 1)
        additional_lines: List of dicts with 'confidence' and 'label' for extra lines
        nbins: Number of bins for the reliability diagram
        plot_title: Title of the plot
        save_path: File path to save the plot (if None, displays it)
    """
    if additional_lines is None:
        additional_lines = []

    fig, ax = plt.subplots(figsize=(10, 8))
    
    total_lines = len(confidence_cols) + len(additional_lines)
    colors = sns.color_palette('husl', total_lines)

    correctness = df[correctness_col].values  # Same correctness for all

    # Plot for confidence columns in the dataframe
    for i, confidence_col in enumerate(confidence_cols):
        confidence = df[confidence_col].values
        bin_conf_means, bin_acc_means, ece = compute_calibration_bins(confidence, correctness, nbins)
        display_label = confidence_col.replace('_', ' ')
        # if baseline, display as conf_sem_ent
        if confidence_col == 'baseline':
            display_label = 'Conf_SemEnt'
        if confidence_col == 'with grounding model':
            display_label = 'Conf (Ours)'
        ax.plot(
            bin_conf_means, bin_acc_means,
            marker='o', linestyle='-', color=colors[i],
            label=f'{display_label} (ECE={ece:.3f})',
            zorder=2
        )
        ax.fill_between(
            bin_conf_means, bin_acc_means, bin_conf_means,
            color=colors[i], alpha=0.1, zorder=1
        )

    # Plot for additional lines
    for j, line in enumerate(additional_lines):
        confidence = line['confidence']
        label = line['label']
        bin_conf_means, bin_acc_means, ece = compute_calibration_bins(confidence, correctness, nbins)
        ax.plot(
            bin_conf_means, bin_acc_means,
            marker='o', linestyle='-', color=colors[len(confidence_cols) + j],
            label=f'{label} (ECE={ece:.3f})',
            zorder=2
        )
        ax.fill_between(
            bin_conf_means, bin_acc_means, bin_conf_means,
            color=colors[len(confidence_cols) + j], alpha=0.1, zorder=1
        )

    # Perfect calibration line
    ax.plot([0, 1], [0, 1], '--', color='black', linewidth=2, label='Perfect Calibration', zorder=2)

    # Plot aesthetics
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.set_xlabel('Confidence', fontsize=20, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=20, fontweight='bold')
    ax.set_title(f'Reliability Diagram', fontsize=20, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='best', fontsize=20)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

# Example usage with more curve for "our metric"
np.random.seed(42)
n_samples = 1000
p_i = np.random.uniform(0, 1, n_samples)
correctness = (np.random.rand(n_samples) < p_i).astype(int)
df = pd.DataFrame({
    'baseline': np.random.uniform(0, 1, n_samples),
    'with grounding model': np.clip(p_i + np.random.normal(0, 0.15, n_samples), 0, 1),
    'correctness': correctness
})

# additional_lines = [
#     {'confidence': np.random.beta(5, 1, size=n_samples), 'label': 'Overconfident Baseline'}
# ]

confidence_cols = ['baseline', 'with grounding model']

plot_multiple_calibration_curve(
    df,
    confidence_cols=confidence_cols,
    correctness_col='correctness',
    additional_lines=None,
    nbins=10,
    plot_title='Calibration with Our Metric',
    save_path='calibration_plot_curved.png'  # Set to None to display
)